# Building MCP Servers: Resources, Tools, and Prompts

An MCP server is a lightweight service that exposes capabilities to LLM hosts. Building a well-designed server requires understanding the three primitive types, their semantics, and how to handle errors correctly.

## Server Primitive Design

**Resources** are the read-only interface to data. They should be:
- Idempotent: reading the same URI always returns the same or current data
- URI-addressable: every piece of data has a stable identifier
- MIME-typed: the client knows how to interpret the content

**Tools** are the action interface. They should be:
- Single-responsibility: one clear action per tool
- Explicitly schema-typed: inputs must match the declared schema
- Error-documented: what errors can this tool return and why?

**Prompts** are reusable templates that reduce prompt engineering on the client side. They take arguments and produce ready-to-use prompt strings.

In [ ]:
import os, json
from dataclasses import dataclass, field
from typing import Callable

class FilesystemMCPServer:
    def __init__(self, root_dir: str):
        self.root = os.path.abspath(root_dir)
        self.server = MCPServer('filesystem-server', '1.0.0')
        self._register()

    def _safe_path(self, path: str) -> tuple:
        full = os.path.normpath(os.path.join(self.root, path.lstrip('/')))
        if not full.startswith(self.root):
            return None, 'Path traversal attempt blocked'
        return full, None

    def _register(self):
        # Register tools
        self.server.add_tool(MCPTool(
            name='read_file',
            description='Read the contents of a file. Path is relative to workspace root.',
            input_schema={'type': 'object', 'properties': {'path': {'type': 'string'}}, 'required': ['path']},
            fn=self._read_file
        ))
        self.server.add_tool(MCPTool(
            name='list_directory',
            description='List files in a directory. Path is relative to workspace root.',
            input_schema={'type': 'object', 'properties': {'path': {'type': 'string', 'default': '/'}}, 'required': []},
            fn=self._list_dir
        ))
        self.server.add_tool(MCPTool(
            name='write_file',
            description='Write content to a file. Creates the file if it does not exist.',
            input_schema={'type': 'object', 'properties': {'path': {'type': 'string'}, 'content': {'type': 'string'}}, 'required': ['path', 'content']},
            fn=self._write_file
        ))
        # Register prompt
        self.server.add_prompt(MCPPrompt(
            name='code_review',
            description='Review code in a file for quality and correctness',
            arguments=[{'name': 'file_path', 'required': True}],
            template='Review the code in {file_path} for: correctness, style, security issues, and performance. Provide specific suggestions.'
        ))

    def _read_file(self, path: str) -> str:
        full, err = self._safe_path(path)
        if err: raise ValueError(err)
        if not os.path.exists(full):
            raise FileNotFoundError(f'File not found: {path}')
        with open(full) as f:
            return f.read()

    def _list_dir(self, path: str = '/') -> str:
        full, err = self._safe_path(path)
        if err: raise ValueError(err)
        if not os.path.isdir(full):
            raise NotADirectoryError(f'Not a directory: {path}')
        entries = os.listdir(full)
        return json.dumps(entries)

    def _write_file(self, path: str, content: str) -> str:
        full, err = self._safe_path(path)
        if err: raise ValueError(err)
        os.makedirs(os.path.dirname(full), exist_ok=True)
        with open(full, 'w') as f:
            f.write(content)
        return f'Written {len(content)} bytes to {path}'

    def handle(self, method: str, params: dict) -> dict:
        return self.server.handle(method, params)

# Demo with /tmp as workspace
fs_server = FilesystemMCPServer('/tmp')
print('Tools available:')
result = fs_server.handle('tools/list', {})
for t in result['tools']:
    print(f'  {t["name"]}: {t["description"]}')

# Write a test file and read it back
fs_server.handle('tools/call', {'name': 'write_file', 'arguments': {'path': 'test_mcp.txt', 'content': 'Hello from MCP!'}})
read_result = fs_server.handle('tools/call', {'name': 'read_file', 'arguments': {'path': 'test_mcp.txt'}})
print(f'\nRead back: {read_result}')

# Test path traversal protection
bad = fs_server.handle('tools/call', {'name': 'read_file', 'arguments': {'path': '../../etc/passwd'}})
print(f'Path traversal attempt: {bad}')

## Error Handling in MCP Servers

MCP defines standard JSON-RPC error codes:
- `-32700`: Parse error
- `-32600`: Invalid request
- `-32601`: Method not found / resource not found
- `-32602`: Invalid parameters
- `-32603`: Internal error

Beyond these, tools should return domain-specific errors in the content field (not as JSON-RPC errors) so the LLM can reason about what went wrong and how to recover. A file-not-found error in the content field lets the model try an alternative path; a JSON-RPC error terminates the tool call.

## Performance Considerations

For production MCP servers:
- **Caching**: expensive operations (database queries, API calls) should be cached with TTL
- **Streaming**: large resources should support streaming to avoid memory overhead
- **Rate limiting**: cap calls per client per second to prevent runaway agent loops
- **Connection pooling**: reuse database and API connections across tool calls
- **Timeouts**: every tool call must have a timeout; unbounded tool execution can stall the entire agent